In [ ]:
from google import colab
colab.drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!unzip "/content/drive/MyDrive/Colab Notebooks/Projects/Final Year Project/Tomato.zip" -d "/content/Dataset"

Streaming output truncated to the last 5000 lines.
  inflating: /content/Dataset/Tomato/Unknown/image3052.jpg  
  inflating: /content/Dataset/Tomato/Unknown/image3055.jpg  
  inflating: /content/Dataset/Tomato/Unknown/image3056.jpg  
  inflating: /content/Dataset/Tomato/Unknown/image3057.jpg  
  inflating: /content/Dataset/Tomato/Unknown/image3059.jpg  
  inflating: /content/Dataset/Tomato/Unknown/image306.jpg  
  inflating: /content/Dataset/Tomato/Unknown/image3060.jpg  
  inflating: /content/Dataset/Tomato/Unknown/image3061.jpg  
  inflating: /content/Dataset/Tomato/Unknown/image3063.jpg  
  inflating: /content/Dataset/Tomato/Unknown/image3064.jpg  
  inflating: /content/Dataset/Tomato/Unknown/image3065.jpg  
  inflating: /content/Dataset/Tomato/Unknown/image3066.jpg  
  inflating: /content/Dataset/Tomato/Unknown/image3067.jpg  
  inflating: /content/Dataset/Tomato/Unknown/image3068.jpg  
  inflating: /content/Dataset/Tomato/Unknown/image3069.jpg  
  inflating: /content/Dataset/Tomat

In [ ]:

import os

DATASET_CANDIDATES = [
    "/content/Dataset/Tomato",
    "/content/Tomato",
]

dataset_path = next((path for path in DATASET_CANDIDATES if os.path.isdir(path)), None)

if dataset_path is None:
    raise FileNotFoundError(
        "Tomato dataset folder not found. Expected one of: "
        + ", ".join(DATASET_CANDIDATES)
    )

print(f"Using dataset path: {dataset_path}")


In [ ]:
# # Optimization: Copy dataset from Drive to local storage to speed up I/O
# import shutil

# local_dataset_path = '/content/Dataset'
# if not os.path.exists(local_dataset_path):
#     print('Copying dataset to local runtime... This takes a few minutes but makes training 10x faster.')
#     shutil.copytree(dataset_path, local_dataset_path)
#     print('Copy complete.')

# # Update dataset_path to local
# dataset_path = local_dataset_path

In [ ]:
from PIL import Image
import os

deleted = 0

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        path = os.path.join(root, file)

        try:
            img = Image.open(path)
            img.verify()
        except:
            os.remove(path)
            deleted += 1
            print("Deleted:", path)

print("Total deleted:", deleted)

Total deleted: 0


In [ ]:
from PIL import Image
import os

deleted = 0

for root, dirs, files in os.walk(dataset_path):

    for file in files:

        path = os.path.join(root, file)

        try:
            img = Image.open(path)
            img.load()

        except Exception as e:
            print("Deleting:", path)

            try:
                os.remove(path)
                deleted += 1
            except:
                pass

print("Total deleted:", deleted)

Total deleted: 0


In [ ]:
import os

for folder in os.listdir(dataset_path):

    folder_path = os.path.join(dataset_path, folder)

    if os.path.isdir(folder_path):

        total = 0

        for root, dirs, files in os.walk(folder_path):
            total += len(files)

        print(folder, ":", total)

Tomato leaf blight : 1288
Unknown : 2000
Tomato healthy : 466
Tomato septoria leaf spot : 2743
Tomato leaf curl : 511
Tomato verticulium wilt : 772


In [ ]:

import os
import random

# =========================================
# SETTINGS
# =========================================

unknown_folder = os.path.join(dataset_path, "Unknown")
target_count = 2000
SEED = 42

if not os.path.isdir(unknown_folder):
    raise FileNotFoundError(f"Unknown class folder not found: {unknown_folder}")

# =========================================
# LOAD FILES
# =========================================

random.seed(SEED)

all_files = [
    f for f in os.listdir(unknown_folder)
    if os.path.isfile(os.path.join(unknown_folder, f))
]

print(f"Original Unknown count: {len(all_files)}")

# =========================================
# RANDOMLY REMOVE EXTRA FILES
# =========================================

if len(all_files) > target_count:
    files_to_remove = random.sample(all_files, len(all_files) - target_count)

    for file_name in files_to_remove:
        file_path = os.path.join(unknown_folder, file_name)
        os.remove(file_path)

    print(f"Trimmed Unknown class to {target_count}")
else:
    print("Unknown class already within limit.")

# =========================================
# FINAL COUNT
# =========================================

final_count = len([
    f for f in os.listdir(unknown_folder)
    if os.path.isfile(os.path.join(unknown_folder, f))
])

print(f"Final Unknown count: {final_count}")


Original Unknown count: 6402
Trimmed Unknown class to 2000
Final Unknown count: 2000


In [ ]:
!pip install tensorflow

In [ ]:

import json
import os
import shutil
import zipfile

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetV2B1
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix

import numpy as np


In [ ]:
# Enable Mixed Precision for faster training on T4/A100 GPUs
from tensorflow.keras import mixed_precision

try:
    policy = mixed_precision.Policy('mixed_float16')
    mixed_precision.set_global_policy(policy)
    print('Mixed precision enabled: ', policy.compute_dtype)
except:
    print('Mixed precision not supported or failed to initialize.')

Mixed precision enabled:  float16


In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS_HEAD = 5
EPOCHS_FINE = 10
SEED = 42
AUTO_RESUME = True
RESUME_PREFERENCE = "last"
CONFIDENCE_THRESHOLDS = [0.70, 0.75, 0.80, 0.85, 0.90]
MARGIN_THRESHOLD = 0.10

In [ ]:

ARTIFACT_ROOT = "/content" if os.path.exists("/content") else "."
RUN_NAME = "tomato_disease_efficientnetv2b1"
RUN_DIR = os.path.join(ARTIFACT_ROOT, RUN_NAME)

RECOVERY_DIR = os.path.join(RUN_DIR, "recovery")
FINAL_DIR = os.path.join(RUN_DIR, "final")
CHECKPOINT_DIR = os.path.join(RUN_DIR, "checkpoints")
CSV_LOG_DIR = os.path.join(RUN_DIR, "logs", "csv")
TENSORBOARD_LOG_DIR = os.path.join(RUN_DIR, "logs", "tensorboard")
BACKUP_ROOT_DIR = os.path.join(RUN_DIR, "backup")
EXPORT_DIR = os.path.join(RUN_DIR, "export")

LAST_MODEL_PATH = os.path.join(RECOVERY_DIR, "last_model.keras")
FINAL_MODEL_PATH = os.path.join(FINAL_DIR, "final_model.keras")
BEST_MODEL_PATH = os.path.join(FINAL_DIR, "best_model.keras")
TOMATO_MODEL_EXPORT_PATH = os.path.join(EXPORT_DIR, "tomato_disease_model.keras")
TOMATO_LABELS_EXPORT_PATH = os.path.join(EXPORT_DIR, "tomato_disease_labels.json")
EPOCH_CHECKPOINT_PATTERN = os.path.join(CHECKPOINT_DIR, "tomato_disease_model_epoch_{epoch:02d}.keras")

for path in [
    RUN_DIR,
    RECOVERY_DIR,
    FINAL_DIR,
    CHECKPOINT_DIR,
    CSV_LOG_DIR,
    TENSORBOARD_LOG_DIR,
    BACKUP_ROOT_DIR,
    EXPORT_DIR,
]:
    os.makedirs(path, exist_ok=True)

print(f"Artifacts directory: {RUN_DIR}")
print(f"Recovery model path: {LAST_MODEL_PATH}")
print(f"Final model path: {FINAL_MODEL_PATH}")
print(f"Export model path: {TOMATO_MODEL_EXPORT_PATH}")


Artifacts directory: ./crop_classifier_efficientnetv2b1
Recovery model path: ./crop_classifier_efficientnetv2b1/recovery/last_model.keras
Final model path: ./crop_classifier_efficientnetv2b1/final/final_model.keras


In [ ]:
train_datagen = ImageDataGenerator(
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.2,
    brightness_range=[0.6, 1.4],
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=SEED
)

val_generator = val_datagen.flow_from_directory(
    dataset_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=SEED
)

class_names = list(train_generator.class_indices.keys())
print(train_generator.class_indices)

Found 6226 images belonging to 6 classes.
Found 1554 images belonging to 6 classes.
{'Tomato healthy': 0, 'Tomato leaf blight': 1, 'Tomato leaf curl': 2, 'Tomato septoria leaf spot': 3, 'Tomato verticulium wilt': 4, 'Unknown': 5}


In [ ]:
base_model = EfficientNetV2B1(
    include_top=False,
    weights='imagenet',
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False

model = keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.2),
    # Output layer
    layers.Dense(
        train_generator.num_classes,
        activation='softmax',
        dtype='float32' # Explicitly use float32 for the output to support all metrics
    )
])

28456008/28456008 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.TopKCategoricalAccuracy(k=2, name="top_2_accuracy"),
    ]
)

In [ ]:

def get_stage_csv_path(stage_name):
    return os.path.join(CSV_LOG_DIR, f"{stage_name}.csv")


def get_completed_epochs(stage_name):
    csv_path = get_stage_csv_path(stage_name)
    if not os.path.exists(csv_path):
        return 0

    with open(csv_path, "r", encoding="utf-8") as f:
        completed = max(sum(1 for _ in f) - 1, 0)

    return completed


def assert_complete_keras_model(model_path):
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Saved model not found: {model_path}")

    with zipfile.ZipFile(model_path) as archive:
        names = set(archive.namelist())

    if "config.json" not in names:
        raise ValueError(f"Invalid .keras model with no config.json: {model_path}")

    if not ({"model.weights.h5", "model.weights.npz"} & names):
        raise ValueError(
            f"Incomplete .keras model with no weights file: {model_path}. "
            "Do not deploy this artifact. Re-run model.save(...) after training."
        )


def save_model_checked(model, model_path):
    model.save(model_path)
    assert_complete_keras_model(model_path)


def save_label_metadata():
    labels = list(class_names)
    metadata = {
        "class_names": labels,
        "class_indices": train_generator.class_indices,
        "num_classes": train_generator.num_classes,
    }

    with open(TOMATO_LABELS_EXPORT_PATH, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    print(f"Saved Tomato label metadata: {TOMATO_LABELS_EXPORT_PATH}")


def save_recovery_snapshot(model, tag=None):
    save_model_checked(model, LAST_MODEL_PATH)
    save_model_checked(model, FINAL_MODEL_PATH)
    save_model_checked(model, TOMATO_MODEL_EXPORT_PATH)
    save_label_metadata()

    if tag:
        tagged_path = os.path.join(RECOVERY_DIR, f"{tag}.keras")
        save_model_checked(model, tagged_path)
        print(f"Saved tagged recovery model: {tagged_path}")

    print(f"Saved latest recovery model: {LAST_MODEL_PATH}")
    print(f"Saved current final model: {FINAL_MODEL_PATH}")
    print(f"Saved deployable Tomato model: {TOMATO_MODEL_EXPORT_PATH}")


def build_callbacks(stage_name):
    stage_backup_dir = os.path.join(BACKUP_ROOT_DIR, stage_name)
    stage_tensorboard_dir = os.path.join(TENSORBOARD_LOG_DIR, stage_name)
    stage_csv_path = get_stage_csv_path(stage_name)

    os.makedirs(stage_backup_dir, exist_ok=True)
    os.makedirs(stage_tensorboard_dir, exist_ok=True)

    return [
        tf.keras.callbacks.BackupAndRestore(
            backup_dir=stage_backup_dir
        ),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=BEST_MODEL_PATH,
            monitor="val_loss",
            mode="min",
            save_best_only=True,
            save_weights_only=False,
            verbose=1
        ),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=EPOCH_CHECKPOINT_PATTERN,
            save_best_only=False,
            save_weights_only=False,
            verbose=1
        ),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=LAST_MODEL_PATH,
            save_best_only=False,
            save_weights_only=False,
            verbose=0
        ),
        tf.keras.callbacks.CSVLogger(
            filename=stage_csv_path,
            append=os.path.exists(stage_csv_path)
        ),
        tf.keras.callbacks.TensorBoard(
            log_dir=stage_tensorboard_dir,
            histogram_freq=0,
            write_graph=False,
            update_freq="epoch",
            profile_batch=0
        ),
        tf.keras.callbacks.TerminateOnNaN(),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.3,
            patience=2,
            min_lr=1e-6,
            verbose=1
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=4,
            restore_best_weights=True,
            verbose=1
        ),
    ]


In [ ]:
def resolve_resume_model_path(prefer="last"):
    candidates = {
        "last": LAST_MODEL_PATH,
        "best": BEST_MODEL_PATH,
        "final": FINAL_MODEL_PATH,
    }

    ordered_paths = [candidates.get(prefer)] + [
        path for name, path in candidates.items() if name != prefer
    ]

    for path in ordered_paths:
        if path and os.path.exists(path):
            return path

    return None


def load_resume_model(prefer="last"):
    resume_path = resolve_resume_model_path(prefer=prefer)
    if resume_path is None:
        print("No saved model artifact found. Starting fresh.")
        return None

    print(f"Loading resume model from: {resume_path}")
    return keras.models.load_model(resume_path)

In [ ]:
# if AUTO_RESUME:
#     resumed_model = load_resume_model(prefer=RESUME_PREFERENCE)
#     if resumed_model is not None:
#         model = resumed_model
#         print("Resume model loaded into current session.")
#     else:
#         print("No resume artifact found. Using freshly initialized model.")
# else:
#     print("AUTO_RESUME is disabled. Using freshly initialized model.")

Loading resume model from: ./crop_classifier_efficientnetv2b1/recovery/last_model.keras
Resume model loaded into current session.


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'loss_scale_optimizer', because it has 14 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 10 variables whereas the saved optimizer has 0 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [ ]:
# # Delete the model and clear the Keras session to free up GPU memory
# if 'model' in globals():
#     del model

# tf.keras.backend.clear_session()
# import gc
# gc.collect()

# print("Model deleted and session cleared.")

Model deleted and session cleared.


In [ ]:
head_stage_name = "frozen_head"
head_callbacks = build_callbacks(head_stage_name)
head_initial_epoch = get_completed_epochs(head_stage_name)
head_target_epoch = EPOCHS_HEAD

history = None

if head_initial_epoch >= head_target_epoch:
    print(f"Frozen-head stage already complete through epoch: {head_initial_epoch}")
else:
    try:
        history = model.fit(
            train_generator,
            validation_data=val_generator,
            initial_epoch=head_initial_epoch,
            epochs=head_target_epoch,
            callbacks=head_callbacks
        )
    except KeyboardInterrupt:
        print("\nFrozen-head training interrupted. Saving recovery artifacts...")
        save_recovery_snapshot(model, tag="frozen_head_interrupt")
    finally:
        save_recovery_snapshot(model)

head_completed_epochs = get_completed_epochs(head_stage_name)
print(f"Frozen-head stage complete through epoch: {head_completed_epochs}")

Epoch 2/5
195/195 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.7413 - loss: 0.6348 - top_2_accuracy: 0.8969
Epoch 2: saving model to ./crop_classifier_efficientnetv2b1/recovery/last_model.keras

Epoch 2: finished saving model to ./crop_classifier_efficientnetv2b1/recovery/last_model.keras
195/195 ━━━━━━━━━━━━━━━━━━━━ 2057s 11s/step - accuracy: 0.7441 - loss: 0.6349 - top_2_accuracy: 0.8972 - val_accuracy: 0.6744 - val_loss: 0.8282 - val_top_2_accuracy: 0.8700 - learning_rate: 0.0010
Epoch 3/5
195/195 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.7510 - loss: 0.6135 - top_2_accuracy: 0.9067
Epoch 3: saving model to ./crop_classifier_efficientnetv2b1/recovery/last_model.keras

Epoch 3: finished saving model to ./crop_classifier_efficientnetv2b1/recovery/last_model.keras
195/195 ━━━━━━━━━━━━━━━━━━━━ 2035s 10s/step - accuracy: 0.7615 - loss: 0.5979 - top_2_accuracy: 0.9099 - val_accuracy: 0.6371 - val_loss: 0.8682 - val_top_2_accuracy: 0.8629 - learning_rate: 0.0010
Epoch 4/5
195/195 ━━━━

In [ ]:

# Export is handled in the final cell after fine-tuning and evaluation.
# Running the old export here would package only the frozen-head checkpoint.


Done! Downloading final_model_export.zip...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
fine_stage_name = "fine_tune"
# Ensure head_completed_epochs is recovered from logs
head_completed_epochs = get_completed_epochs("frozen_head")

fine_completed_only = get_completed_epochs(fine_stage_name)
fine_initial_epoch = head_completed_epochs + fine_completed_only
fine_target_epoch = head_completed_epochs + EPOCHS_FINE
fine_callbacks = build_callbacks(fine_stage_name)

base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.TopKCategoricalAccuracy(k=2, name="top_2_accuracy"),
    ]
)

history_fine = None

if fine_initial_epoch >= fine_target_epoch:
    print(f"Fine-tuning stage already complete through epoch: {fine_initial_epoch}")
else:
    try:
        history_fine = model.fit(
            train_generator,
            validation_data=val_generator,
            initial_epoch=fine_initial_epoch,
            epochs=fine_target_epoch,
            callbacks=fine_callbacks
        )
    except KeyboardInterrupt:
        print("\nFine-tuning interrupted. Saving recovery artifacts...")
        save_recovery_snapshot(model, tag="fine_tune_interrupt")
    finally:
        save_recovery_snapshot(model)

fine_completed_epochs = get_completed_epochs(fine_stage_name)
print(f"Fine-tuning stage complete through fine-tune epoch count: {fine_completed_epochs}")

Epoch 2/15
195/195 ━━━━━━━━━━━━━━━━━━━━ 0s 25s/step - accuracy: 0.7558 - loss: 0.6647 - top_2_accuracy: 0.9001 
Epoch 2: saving model to ./crop_classifier_efficientnetv2b1/recovery/last_model.keras

Epoch 2: finished saving model to ./crop_classifier_efficientnetv2b1/recovery/last_model.keras
195/195 ━━━━━━━━━━━━━━━━━━━━ 5276s 27s/step - accuracy: 0.7610 - loss: 0.6505 - top_2_accuracy: 0.9046 - val_accuracy: 0.6577 - val_loss: 0.8408 - val_top_2_accuracy: 0.8732 - learning_rate: 1.0000e-05
Epoch 3/15
195/195 ━━━━━━━━━━━━━━━━━━━━ 0s 25s/step - accuracy: 0.7915 - loss: 0.5639 - top_2_accuracy: 0.9243 
Epoch 3: saving model to ./crop_classifier_efficientnetv2b1/recovery/last_model.keras

Epoch 3: finished saving model to ./crop_classifier_efficientnetv2b1/recovery/last_model.keras
195/195 ━━━━━━━━━━━━━━━━━━━━ 5226s 27s/step - accuracy: 0.7870 - loss: 0.5700 - top_2_accuracy: 0.9194 - val_accuracy: 0.6583 - val_loss: 0.8388 - val_top_2_accuracy: 0.8732 - learning_rate: 1.0000e-05
Epoch 4/

In [ ]:
val_generator.reset()
val_probs = model.predict(val_generator, verbose=1)
val_true = val_generator.classes
unknown_idx = class_names.index("Unknown")

top1 = np.argmax(val_probs, axis=1)
top1_conf = np.max(val_probs, axis=1)
top2_conf = np.partition(val_probs, -2, axis=1)[:, -2]
margins = top1_conf - top2_conf

print("Validation confusion matrix (raw argmax):")
print(confusion_matrix(val_true, top1))
print()
print(classification_report(val_true, top1, target_names=class_names, digits=4))

for threshold in CONFIDENCE_THRESHOLDS:
    final_pred = top1.copy()
    reject_mask = (top1 == unknown_idx) | (top1_conf < threshold) | (margins < MARGIN_THRESHOLD)
    final_pred[reject_mask] = unknown_idx

    accepted_mask = final_pred != unknown_idx
    accepted_total = int(np.sum(accepted_mask))
    accepted_accuracy = float(np.mean(final_pred[accepted_mask] == val_true[accepted_mask])) if accepted_total else 0.0
    acceptance_rate = float(np.mean(accepted_mask))

    true_unknown_mask = val_true == unknown_idx
    unknown_recall = float(np.mean(final_pred[true_unknown_mask] == unknown_idx))
    false_accept_rate = float(np.mean(final_pred[true_unknown_mask] != unknown_idx))

    print(
        f"threshold={threshold:.2f} | accepted_accuracy={accepted_accuracy:.4f} | "
        f"acceptance_rate={acceptance_rate:.4f} | unknown_recall={unknown_recall:.4f} | "
        f"unknown_false_accept_rate={false_accept_rate:.4f}"
    )


In [ ]:

def export_final_artifacts():
    save_recovery_snapshot(model, tag="manual_final_save")

    zip_base = os.path.join(EXPORT_DIR, "tomato_disease_model_export")
    zip_path = f"{zip_base}.zip"
    temp_dir = os.path.join(EXPORT_DIR, "package")

    if os.path.exists(temp_dir):
        shutil.rmtree(temp_dir)
    os.makedirs(temp_dir, exist_ok=True)

    shutil.copy2(TOMATO_MODEL_EXPORT_PATH, os.path.join(temp_dir, "tomato_disease_model.keras"))
    shutil.copy2(TOMATO_LABELS_EXPORT_PATH, os.path.join(temp_dir, "tomato_disease_labels.json"))

    if os.path.exists(zip_path):
        os.remove(zip_path)
    shutil.make_archive(zip_base, "zip", temp_dir)
    shutil.rmtree(temp_dir)

    print(f"Best model available at: {BEST_MODEL_PATH}")
    print(f"Epoch checkpoints directory: {CHECKPOINT_DIR}")
    print(f"Deployable model available at: {TOMATO_MODEL_EXPORT_PATH}")
    print(f"Export package available at: {zip_path}")

    try:
        from google.colab import files
        files.download(zip_path)
    except Exception:
        print("Not running in Colab, skipping automatic download.")


export_final_artifacts()
